In [ ]:
# Install RF-DETR (official Roboflow package) and supporting libraries
!pip install -q rfdetr roboflow supervision pycocotools matplotlib Pillow
print('✅ All packages installed.')
!pip install -q "rfdetr[train,loggers]"

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
CHECKPOINT_DIR = '/content/drive/MyDrive/CO3_1_RFDETR_checkpoints'
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
print(f'✅ Drive mounted. Checkpoints will save to: {CHECKPOINT_DIR}')

In [ ]:
from roboflow import Roboflow

API_KEY = "B9GFnVJTMeFxb9eci8wC"

rf = Roboflow(API_KEY)

project = rf.workspace("universe-datasets").project("hard-hat-universe-0dy7t")
version = project.version(22)
dataset = version.download("coco")

DATASET_DIR = dataset.location
print(f'✅ Dataset downloaded to: {DATASET_DIR}')

In [ ]:
import os, json
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image
import numpy as np

# Check folder structure
for split in ['train', 'valid', 'test']:
    img_dir = os.path.join(DATASET_DIR, split)
    ann_file = os.path.join(DATASET_DIR, split, '_annotations.coco.json')
    if os.path.exists(img_dir):
        imgs = [f for f in os.listdir(img_dir) if f.endswith(('.jpg', '.png', '.jpeg'))]
        with open(ann_file) as f:
            ann_data = json.load(f)
        print(f'{split:8s} → {len(imgs):5d} images | {len(ann_data["annotations"]):6d} annotations | {len(ann_data["categories"])} classes')

# Print class names
with open(os.path.join(DATASET_DIR, 'train', '_annotations.coco.json')) as f:
    train_ann = json.load(f)
classes = {cat['id']: cat['name'] for cat in train_ann['categories']}
print(f'\n📋 Classes: {list(classes.values())}')

In [ ]:
import random

COLORS = {'head': 'red', 'helmet': 'lime', 'person': 'blue'}

def show_annotated_samples(dataset_dir, split='train', n=4):
    ann_file = os.path.join(dataset_dir, split, '_annotations.coco.json')
    img_dir  = os.path.join(dataset_dir, split)
    with open(ann_file) as f:
        data = json.load(f)
    cat_map = {cat['id']: cat['name'] for cat in data['categories']}
    img_map  = {img['id']: img for img in data['images']}
    ann_map  = {}
    for ann in data['annotations']:
        ann_map.setdefault(ann['image_id'], []).append(ann)

    sample_ids = random.sample(list(img_map.keys()), n)
    fig, axes = plt.subplots(1, n, figsize=(5*n, 5))
    for ax, img_id in zip(axes, sample_ids):
        img_info = img_map[img_id]
        img_path = os.path.join(img_dir, img_info['file_name'])
        img = Image.open(img_path).convert('RGB')
        ax.imshow(img)
        for ann in ann_map.get(img_id, []):
            x, y, w, h = ann['bbox']
            label = cat_map[ann['category_id']]
            color = COLORS.get(label, 'yellow')
            rect = patches.Rectangle((x, y), w, h, linewidth=2, edgecolor=color, facecolor='none')
            ax.add_patch(rect)
            ax.text(x, y - 4, label, color=color, fontsize=9, fontweight='bold')
        ax.axis('off')
        ax.set_title(img_info['file_name'][:25], fontsize=8)
    plt.suptitle(f'Sample Annotated Images — {split} split', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()

show_annotated_samples(DATASET_DIR, split='train', n=4)

In [ ]:
from rfdetr import RFDETRBase

# RFDETRBase = 29M params — fits in free Colab T4 (15GB VRAM)
# RFDETRLarge = 128M params — needs Colab Pro
model = RFDETRBase()

print('✅ RF-DETR Base model loaded.')
print('Architecture: DINOv2 backbone + Detection Transformer')
print('Parameters: ~29M')

In [ ]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

model.train(
    dataset_dir=DATASET_DIR,
    epochs=10,
    batch_size=1,
    grad_accum_steps=16,
    lr=1e-4,
    resolution=560,
    output_dir=CHECKPOINT_DIR,
    early_stopping_patience=5,
)

print('✅ Training complete! Checkpoints saved to Google Drive.')

In [ ]:
import gc, torch
gc.collect()
torch.cuda.empty_cache()

# Re-initialize model with best checkpoint weights
from rfdetr import RFDETRBase
model = RFDETRBase(
    pretrain_weights=f"{CHECKPOINT_DIR}/checkpoint_best_regular.pth",
    num_classes=3
)
print("✅ Best checkpoint loaded!")

In [ ]:
import json, os
import numpy as np
from PIL import Image
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval

TEST_ANN_FILE = os.path.join(DATASET_DIR, 'test', '_annotations.coco.json')
TEST_IMG_DIR  = os.path.join(DATASET_DIR, 'test')

coco_gt = COCO(TEST_ANN_FILE)
with open(TEST_ANN_FILE) as f:
    test_data = json.load(f)

# Map 0-indexed class_id to actual GT category id
id_remap = {cat['id']: cat['id'] for cat in test_data['categories']}
cat_map  = {cat['id']: cat['name'] for cat in test_data['categories']}

results = []
print(f'Running inference on {len(test_data["images"])} test images...')

for img_info in test_data['images']:
    img_path = os.path.join(TEST_IMG_DIR, img_info['file_name'])
    if not os.path.exists(img_path):
        continue
    img = Image.open(img_path).convert('RGB')
    det = model.predict(img, threshold=0.5)

    if det is None or len(det.xyxy) == 0:
        continue

    for i in range(len(det.xyxy)):
        x1, y1, x2, y2 = det.xyxy[i]
        score   = float(det.confidence[i])
        cat_id  = int(det.class_id[i])  # 0-indexed, matches GT directly
        results.append({
            'image_id':    img_info['id'],
            'category_id': cat_id,
            'bbox':        [float(x1), float(y1), float(x2-x1), float(y2-y1)],
            'score':       score
        })

print(f'Total detections: {len(results)}')

if results:
    coco_dt = coco_gt.loadRes(results)
    coco_eval = COCOeval(coco_gt, coco_dt, 'bbox')
    coco_eval.evaluate()
    coco_eval.accumulate()
    coco_eval.summarize()
    map50_95 = coco_eval.stats[0]
    map50    = coco_eval.stats[1]
    print(f'\n📈 mAP@0.50:0.95 = {map50_95:.4f}')
    print(f'📈 mAP@0.50      = {map50:.4f}')
else:
    print('⚠️ No detections found.')

In [ ]:
def compute_iou(box1, box2):
    """Compute IoU between two boxes in [x1, y1, x2, y2] format."""
    x1 = max(box1[0], box2[0])
    y1 = max(box1[1], box2[1])
    x2 = min(box1[2], box2[2])
    y2 = min(box1[3], box2[3])
    inter = max(0, x2 - x1) * max(0, y2 - y1)
    area1 = (box1[2]-box1[0]) * (box1[3]-box1[1])
    area2 = (box2[2]-box2[0]) * (box2[3]-box2[1])
    union = area1 + area2 - inter
    return inter / union if union > 0 else 0

IOU_THRESHOLD = 0.5
class_stats = {name: {'TP': 0, 'FP': 0, 'FN': 0} for name in cat_map.values()}

# Build GT dict
gt_by_img = {}
for ann in test_data['annotations']:
    gt_by_img.setdefault(ann['image_id'], []).append(ann)

# Build pred dict
pred_by_img = {}
for r in results:
    pred_by_img.setdefault(r['image_id'], []).append(r)

for img_info in test_data['images']:
    img_id = img_info['id']
    gts   = gt_by_img.get(img_id, [])
    preds = sorted(pred_by_img.get(img_id, []), key=lambda x: -x['score'])
    matched_gt = set()

    for pred in preds:
        px1, py1, pw, ph = pred['bbox']
        pred_box = [px1, py1, px1+pw, py1+ph]
        pred_cls = pred['category_id']
        pred_name = cat_map.get(pred_cls, 'unknown')
        best_iou, best_gt_idx = 0, -1

        for gi, gt in enumerate(gts):
            if gi in matched_gt or gt['category_id'] != pred_cls:
                continue
            gx, gy, gw, gh = gt['bbox']
            gt_box = [gx, gy, gx+gw, gy+gh]
            iou = compute_iou(pred_box, gt_box)
            if iou > best_iou:
                best_iou, best_gt_idx = iou, gi

        if best_iou >= IOU_THRESHOLD:
            class_stats[pred_name]['TP'] += 1
            matched_gt.add(best_gt_idx)
        else:
            class_stats[pred_name]['FP'] += 1

    # FN: GT boxes not matched
    for gi, gt in enumerate(gts):
        if gi not in matched_gt:
            gt_name = cat_map.get(gt['category_id'], 'unknown')
            class_stats[gt_name]['FN'] += 1

# --- Print results table ---
print(f'\n{"Class":<12} {"TP":>6} {"FP":>6} {"FN":>6} {"Precision":>10} {"Recall":>10} {"F1":>8}')
print('-' * 62)
all_tp = all_fp = all_fn = 0
for cls_name, s in class_stats.items():
    tp, fp, fn = s['TP'], s['FP'], s['FN']
    prec = tp / (tp + fp) if (tp + fp) > 0 else 0
    rec  = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1   = 2*prec*rec / (prec+rec) if (prec+rec) > 0 else 0
    print(f'{cls_name:<12} {tp:>6} {fp:>6} {fn:>6} {prec:>10.4f} {rec:>10.4f} {f1:>8.4f}')
    all_tp += tp; all_fp += fp; all_fn += fn

print('-' * 62)
overall_prec = all_tp / (all_tp + all_fp) if (all_tp + all_fp) > 0 else 0
overall_rec  = all_tp / (all_tp + all_fn) if (all_tp + all_fn) > 0 else 0
overall_f1   = 2*overall_prec*overall_rec / (overall_prec+overall_rec) if (overall_prec+overall_rec) > 0 else 0
accuracy     = all_tp / (all_tp + all_fp + all_fn) if (all_tp + all_fp + all_fn) > 0 else 0
print(f'{"OVERALL":<12} {all_tp:>6} {all_fp:>6} {all_fn:>6} {overall_prec:>10.4f} {overall_rec:>10.4f} {overall_f1:>8.4f}')
print(f'\n🎯 Classification Accuracy (IoU≥0.5): {accuracy:.4f} ({accuracy*100:.2f}%)')
print(f'📈 mAP@0.50 (from COCO eval above)  : {map50:.4f}')

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

cls_names, precisions, recalls, f1s = [], [], [], []
for cls_name, s in class_stats.items():
    tp, fp, fn = s['TP'], s['FP'], s['FN']
    prec = tp / (tp + fp) if (tp + fp) > 0 else 0
    rec  = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1   = 2*prec*rec / (prec+rec) if (prec+rec) > 0 else 0
    cls_names.append(cls_name)
    precisions.append(prec)
    recalls.append(rec)
    f1s.append(f1)

x = np.arange(len(cls_names))
width = 0.25
fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(x - width, precisions, width, label='Precision', color='steelblue')
ax.bar(x,         recalls,    width, label='Recall',    color='darkorange')
ax.bar(x + width, f1s,        width, label='F1 Score',  color='seagreen')
ax.set_xticks(x)
ax.set_xticklabels(cls_names, fontsize=12)
ax.set_ylim(0, 1.1)
ax.set_ylabel('Score')
ax.set_title('Per-Class Precision, Recall & F1 Score — RF-DETR (Hard Hat Universe)', fontsize=13)
ax.legend()
ax.grid(axis='y', alpha=0.3)
for bar in ax.patches:
    ax.annotate(f'{bar.get_height():.2f}',
                (bar.get_x() + bar.get_width()/2, bar.get_height()),
                ha='center', va='bottom', fontsize=9)
plt.tight_layout()
plt.savefig('/content/metrics_chart.png', dpi=150)
plt.show()
print('✅ Chart saved to /content/metrics_chart.png')

In [ ]:
import random
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image

PRED_COLORS = {0: 'red', 1: 'lime', 2: 'dodgerblue'}
CLASS_NAMES = {0: 'Hard-Hats', 1: 'head', 2: 'helmet'}

def visualize_predictions(dataset_dir, model, split='test', n=6, threshold=0.5):
    ann_file = os.path.join(dataset_dir, split, '_annotations.coco.json')
    img_dir  = os.path.join(dataset_dir, split)
    with open(ann_file) as f:
        data = json.load(f)

    sample_imgs = random.sample(data['images'], min(n, len(data['images'])))
    fig, axes = plt.subplots(2, n//2, figsize=(5*(n//2), 10))
    axes = axes.flatten()

    for ax, img_info in zip(axes, sample_imgs):
        img_path = os.path.join(img_dir, img_info['file_name'])
        img = Image.open(img_path).convert('RGB')
        det = model.predict(img, threshold=threshold)
        ax.imshow(img)

        if det is not None and len(det.xyxy) > 0:
            for i in range(len(det.xyxy)):
                x1, y1, x2, y2 = det.xyxy[i]
                score  = float(det.confidence[i])
                cls_id = int(det.class_id[i])
                label  = CLASS_NAMES.get(cls_id, 'unknown')
                color  = PRED_COLORS.get(cls_id, 'yellow')
                rect = patches.Rectangle((x1, y1), x2-x1, y2-y1,
                                          linewidth=2, edgecolor=color, facecolor='none')
                ax.add_patch(rect)
                ax.text(x1, y1-5, f'{label} {score:.2f}',
                        color=color, fontsize=8, fontweight='bold',
                        bbox=dict(facecolor='black', alpha=0.4, pad=1))
        ax.axis('off')
        ax.set_title(img_info['file_name'][:22], fontsize=7)

    plt.suptitle('RF-DETR Predictions — Hard Hat Universe Test Set', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig('/content/predictions_viz.png', dpi=150)
    plt.show()
    print('✅ Visualization saved to /content/predictions_viz.png')

visualize_predictions(DATASET_DIR, model, split='test', n=6, threshold=0.5)

In [ ]:
import shutil

OUTPUT_DIR = '/content/drive/MyDrive/CO3_1_RFDETR_outputs'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Save charts
shutil.copy('/content/metrics_chart.png',    os.path.join(OUTPUT_DIR, 'metrics_chart.png'))
shutil.copy('/content/predictions_viz.png',  os.path.join(OUTPUT_DIR, 'predictions_viz.png'))

# Save metrics summary as txt
summary_path = os.path.join(OUTPUT_DIR, 'metrics_summary.txt')
with open(summary_path, 'w') as f:
    f.write('=== RF-DETR Evaluation Results — Hard Hat Universe ===\n\n')
    f.write(f'mAP@0.50:0.95 = {map50_95:.4f}\n')
    f.write(f'mAP@0.50      = {map50:.4f}\n')
    f.write(f'Accuracy      = {accuracy:.4f}\n')
    f.write(f'Overall Precision = {overall_prec:.4f}\n')
    f.write(f'Overall Recall    = {overall_rec:.4f}\n')
    f.write(f'Overall F1        = {overall_f1:.4f}\n\n')
    f.write('Per-class breakdown:\n')
    for cls_name, s in class_stats.items():
        tp, fp, fn = s['TP'], s['FP'], s['FN']
        prec = tp / (tp + fp) if (tp + fp) > 0 else 0
        rec  = tp / (tp + fn) if (tp + fn) > 0 else 0
        f1   = 2*prec*rec / (prec+rec) if (prec+rec) > 0 else 0
        f.write(f'  {cls_name}: Prec={prec:.4f}, Rec={rec:.4f}, F1={f1:.4f}\n')

print(f'✅ All outputs saved to Google Drive: {OUTPUT_DIR}')
print('Files saved:')
for fname in os.listdir(OUTPUT_DIR):
    print(f'  - {fname}')